In [2]:
#.venv 
prod_connection_string = "DRIVER={ODBC Driver 17 for SQL Server};Server=CUBO-INTERMODA;Database=IMClientesIV;UID=iditm;PWD=Int3r-M0d@.Id@;Trusted_Connection=no;"
url = "https://unikfashiongt.odoo.com"
db = "rocketgithub-unikfashiongt-odoo-sh-main-25251833"
username = "rmartinez@intermoda.com.hn"
password = "Intermod@2026/?"

#Autenticación con Odoo
from datetime import datetime
import xmlrpc.client
import pandas as pd
import pyodbc
import json


common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})

In [3]:
#Obtener las facturas de venta
models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object")
#hoy menos 10 dias
fecha_inicio = datetime.now().date()- pd.Timedelta(days=1)
fecha_inicio = fecha_inicio.strftime('%Y-%m-%d')
invoice = pd.DataFrame(models.execute_kw(db, uid, password, 'pos.order', 'search_read',[[['date_order','>',fecha_inicio]]]))
print(len(invoice))

69


In [4]:
#obtener los clientes
with pyodbc.connect(prod_connection_string) as conn:
    cursor = conn.cursor()
    cursor.execute("EXEC dbo.SP_ObtenerClientes")

    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()

    df = pd.DataFrame.from_records(rows, columns=columns)

    conn.commit()
Clientes = df[df['CodigoCliente'] == "IMGT-000001134"]
#print(Clientes)

In [5]:
def obtener_CodigoDeBarraInfo(codigo_barra):
    with pyodbc.connect(prod_connection_string) as conn:
        cursor = conn.cursor()
        query = "EXEC dbo.SP_GetCodigosDeBarraInfo ?;"
        json_data = '[{"CodigoBarra":' + codigo_barra + '}]'
        cursor.execute(query, json_data )
        columns = [column[0] for column in cursor.description]
        rows = cursor.fetchall()
        df = pd.DataFrame.from_records(rows, columns=columns)
        return df

In [6]:
def enviar_IntermodaUnikFashion(json_data):
    with pyodbc.connect(prod_connection_string) as conn:
        cursor = conn.cursor()
        query = "EXEC dbo.SP_InsertarVentas ?;"
        cursor.execute(query, (json_data,))
        conn.commit()

In [8]:
registros = []
cantidad_lineas = 0

for index, row in invoice.iterrows():
    order_id = row['id']
    products = pd.DataFrame(models.execute_kw(
        db, uid, password,
        'pos.order.line',
        'search_read',
        [[['order_id', '=', order_id]]]
    ))

    tienda = row['config_id'][1] if row['config_id'] else 'Desconocida'

    print(f"Procesando factura: {row['pos_reference']}: {len(products)} productos")

    for index, product in products.iterrows():
        product_odoo = pd.DataFrame(models.execute_kw(
            db, uid, password,
            'product.product',
            'search_read',
            [[['id', '=', product['product_id'][0]]]],
            {'limit': 1}
        )).rename(columns={'barcode': 'CodigoBarra'})

        codigo_barra = (
            product_odoo['CodigoBarra'].values[0]
            if 'CodigoBarra' in product_odoo.columns and len(product_odoo['CodigoBarra']) > 0
            else None
        )

        if pd.isna(codigo_barra) or codigo_barra in (False, '', None):
            continue

        try:
            results = obtener_CodigoDeBarraInfo(codigo_barra)
        except Exception as e:
            print(f"Error al obtener información para el código de barra {codigo_barra}: {e}")
            continue

        if results.empty:
            print(f"No se encontró información para el código de barra {codigo_barra}")
            continue

        fecha_venta_dt = (
            pd.to_datetime(row['date_order']).date()
            if 'date_order' in row and pd.notna(row['date_order'])
            else None
        )

        fecha_venta = fecha_venta_dt.strftime('%Y-%m-%d') if fecha_venta_dt else None

        json_data = {
            "CodigoCliente": Clientes['CodigoCliente'].values[0],
            "Cliente": Clientes['Cliente'].values[0],
            "CodigoTienda": None,
            "Tienda": tienda,
            "Fecha": fecha_venta,
            "Año": fecha_venta_dt.year if fecha_venta_dt else None,
            "Semestre": "S1" if fecha_venta_dt and fecha_venta_dt.month <= 6 else "S2" if fecha_venta_dt else None,
            "Trimestre": (
                "Q1" if fecha_venta_dt and fecha_venta_dt.month <= 3 else
                "Q2" if fecha_venta_dt and fecha_venta_dt.month <= 6 else
                "Q3" if fecha_venta_dt and fecha_venta_dt.month <= 9 else
                "Q4" if fecha_venta_dt else None
            ),
            "NoMes": fecha_venta_dt.month if fecha_venta_dt else None,
            "Mes": fecha_venta_dt.strftime('%B') if fecha_venta_dt else None,
            "Semana": fecha_venta_dt.isocalendar()[1] if fecha_venta_dt else None,
            "DiaSemana": fecha_venta_dt.strftime('%A') if fecha_venta_dt else None,
            "Dia": fecha_venta_dt.day if fecha_venta_dt else None,
            "CodigoBarra": codigo_barra,
            "CodigoArticulo": results['CodigoArticulo'].values[0] if 'CodigoArticulo' in results.columns else None,
            "Descripcion": results['Descripcion'].values[0] if 'Descripcion' in results.columns else None,
            "CodigoColor": results['CodigoColor'].values[0] if 'CodigoColor' in results.columns else None,
            "Color": results['Color'].values[0] if 'Color' in results.columns else None,
            "Talla": results['Talla'].values[0] if 'Talla' in results.columns else None,
            "Linea": results['Linea'].values[0] if 'Linea' in results.columns else None,
            "Sublinea": results['Sublinea'].values[0] if 'Sublinea' in results.columns else None,
            "Categoria": results['Categoria'].values[0] if 'Categoria' in results.columns else None,
            "Base": results['Base'].values[0] if 'Base' in results.columns else None,
            "Genero": results['Genero'].values[0] if 'Genero' in results.columns else None,
            "Clasificacion": results['Clasificacion'].values[0] if 'Clasificacion' in results.columns else None,
            "LoteOrigen": results['LoteOrigen'].values[0] if 'LoteOrigen' in results.columns else None,
            "Costo": None,
            "Precio": product['price_unit'],
            "Cantidad": float(product['qty']),
            "Ganacia": None,
            "ImporteTotal": (
                product['price_unit'] * product['qty']
                if pd.notna(product['price_unit']) and pd.notna(product['qty'])
                else None
            ),
            "GananciaTotal": None,
            "CostoTotal": None
        }

        registros.append(json_data)
        cantidad_lineas += 1
df = pd.DataFrame(registros)

columnas_sumar = ["Cantidad", "ImporteTotal", "GananciaTotal", "CostoTotal"]

columnas_agrupar = [
    col for col in df.columns
    if col not in columnas_sumar
]

df_agrupado = (
    df.groupby(columnas_agrupar, dropna=False, as_index=False)[columnas_sumar]
    .sum()
)

print(f"Total de registros procesados: {cantidad_lineas}")
print(f"Total de registros agrupados: {len(df_agrupado)}")

#
#for _, row in df_agrupado.iterrows():
#    json_data = json.dumps(row.to_dict(), ensure_ascii=False)
#    enviar_IntermodaUnikFashion(json_data)
#
print("Proceso completado")

Procesando factura: Orden 00848-004-0006: 1 productos
Procesando factura: Orden 00861-002-0004: 6 productos
Procesando factura: Orden 00841-004-0005: 4 productos
Procesando factura: Orden 00858-002-0002: 2 productos
Procesando factura: Orden 00853-005-0007: 2 productos
Procesando factura: Orden 00853-004-0006: 1 productos
Procesando factura: Orden 00853-001-0005: 5 productos
Procesando factura: Orden 00858-001-0001: 3 productos
Procesando factura: Orden 00848-003-0005: 3 productos
Procesando factura: Orden 00854-001-0001: 2 productos
Procesando factura: Orden 00856-001-0002: 1 productos
Procesando factura: Orden 00841-004-0004: 3 productos
Procesando factura: Orden 00841-004-0003: 1 productos
Procesando factura: Orden 00855-001-0005: 2 productos
Procesando factura: Orden 00841-003-0002: 2 productos
Procesando factura: Orden 00852-001-0008: 3 productos
Procesando factura: Orden 00857-001-0003: 1 productos
Procesando factura: Orden 00853-001-0004: 6 productos
Procesando factura: Orden 00